### Alerta de ordens pendentes - Criação de KPI:

Criar tabela de Ordens Pendentes unindo a quantidade de itens e informações de cliente para o envio alerta por e-mail e telefone, para quem possui o cadastro completo

In [0]:
%python
# Definir pastas do projeto em variáveis para facilitar
bronze_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/'
silver_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/'
gold_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/gold/'
resource_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem'
resource_path_volume = '/Volumes/bikestore/logistics/bikestore_resource/origem/'

In [0]:
%python
# Criando várias tabelas temporárias 
silver_map = {
    'tmp_silver_customer': f'{silver_path}/customer/',
    'tmp_silver_orders': f'{silver_path}/orders/',
    'tmp_silver_product': f'{silver_path}/product/'
}
for view_name, path in silver_map.items():
    (spark.read.format('delta')
    .load(path)
    .createOrReplaceTempView(view_name)
) 

In [0]:
SELECT * FROM tmp_silver_customer

In [0]:
DESCRIBE tmp_silver_customer

In [0]:
-- Realizando o CTE
WITH pending AS (
SELECT
customer_id,
order_date,
SUM(quantity) AS quantity,
store_name
status
FROM tmp_silver_orders
WHERE status = 'Pending'
GROUP BY customer_id, order_date, store_name, status
)

SELECT
p.*,
C.first_name AS first_name_customer,
C.email,
C.phone
FROM pending AS P
LEFT JOIN tmp_silver_customer AS C ON P.customer_id = C.customer_id
WHERE C.email IS NOT NULL
AND C.phone IS NOT NULL

In [0]:
DESCRIBE tmp_silver_orders

In [0]:
%python
# Salvando no formato delta na camada Gold
df_orders_pending = spark.sql("""
-- Realizando o CTE
WITH pending AS (
SELECT
customer_id,
order_date,
SUM(quantity) AS quantity,
store_name
status
FROM tmp_silver_orders
WHERE status = 'Pending'
GROUP BY customer_id, order_date, store_name, status
)

SELECT
p.*,
C.first_name AS first_name_customer,
C.email,
C.phone
FROM pending AS P
LEFT JOIN tmp_silver_customer AS C ON P.customer_id = C.customer_id
WHERE C.email IS NOT NULL
AND C.phone IS NOT NULL                           
""")

# Salvar e transformar em formato delta na camada Gold
df_orders_pending.write \
    .mode('overwrite') \
    .format('delta') \
    .option('mergeSchema', 'true') \
    .save(f'{gold_path}/orders_pending')

In [0]:
%sql
/* Criar tabela física de produção, este comando precisa rodar só 1 vez, pois quando os arquivos forem atualizados a tabela refletirá o resultado (Vai obter o log mais recente e trazer dados do(s) arquivo(s) atualizado)*/
CREATE TABLE IF NOT EXISTS bikestore.logistics.gold_orders_pending
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/gold/orders_pending';

In [0]:
SELECT * FROM bikestore.logistics.gold_orders_pending;